# 1. Imports & Loading Aligned Data and Embeddings

We load:
1. **`raw_data` (`fra_perfumes.csv`)**: Filtered with `.dropna(subset=["Description"]).reset_index(drop=True)` so its row positions (`0..N-1`) match `data.pkl` and the `.npy` embedding arrays 1-to-1. We use `raw_data` at display time to show the full original `Description` and `Main Accords`.
2. **`data` (`data.pkl`)**: The cleaned DataFrame containing `Name`, `Brand`, `Gender`, and the Bayesian `Weighted Rating`.
3. **Precomputed Embeddings**: `desc_embs.npy`, `accord_embs.npy`, `gender_embss.npy`, and `brand_embs.npy`.

> **Backward-Compatibility Safeguard**: If `gender_embss.npy` on disk was generated before the `men`/`women` mapping fix (detected by non-zero values in the 3rd column), we automatically rebuild `gender_embs` in memory from `data["Gender"]` so gender matching works immediately.

In [ ]:
import re
import pickle
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize

In [ ]:
# Drop missing descriptions and reset index on both DataFrames so positional index 0..N-1 aligns everywhere
raw_data = pd.read_csv("fra_perfumes.csv").dropna(subset=["Description"]).reset_index(drop=True)

with open("data.pkl", "rb") as f:
    data = pickle.load(f).reset_index(drop=True)

print(f"Loaded {len(data)} perfumes (raw_data aligned: {len(raw_data) == len(data)})")

In [ ]:
UNISEX_VEC = np.array([1 / np.sqrt(2), 1 / np.sqrt(2)], dtype=np.float32)
GENDER_VECTORS = {
    "men":    np.array([1.0, 0.0], dtype=np.float32),
    "male":   np.array([1.0, 0.0], dtype=np.float32),
    "women":  np.array([0.0, 1.0], dtype=np.float32),
    "female": np.array([0.0, 1.0], dtype=np.float32),
    "unisex": UNISEX_VEC,
}

def expand_gender_vec(gv, dim=384):
    """Zero-pad 2D gender vector to 384 dimensions to match MiniLM embedding size."""
    return np.pad(gv, (0, dim - len(gv)), mode="constant").astype(np.float32)

desc_embs = np.load("desc_embs.npy").astype(np.float32)
accord_embs = np.load("accord_embs.npy").astype(np.float32)
gender_embs = np.load("gender_embss.npy").astype(np.float32)
brand_embs = np.load("brand_embs.npy").astype(np.float32)

# Auto-repair gender_embs if the file on disk still has the legacy [0, 0, 1] fallback
if np.any(gender_embs[:, 2] != 0):
    gender_embs = np.vstack([
        expand_gender_vec(GENDER_VECTORS.get(str(g).strip().lower(), UNISEX_VEC))
        for g in data["Gender"]
    ]).astype(np.float32)
    gender_embs = normalize(gender_embs).astype(np.float32)

# 2. Multi-Modal Embedding Fusion & FAISS Indexing

We combine the four L2-normalized feature embeddings into a single unified 384-dimensional vector per perfume:
$$\mathbf{e}_{\text{final}} = \text{normalize}\left(0.45\,\mathbf{e}_{\text{desc}} + 0.25\,\mathbf{e}_{\text{accord}} + 0.20\,\mathbf{e}_{\text{gender}} + 0.10\,\mathbf{e}_{\text{brand}}\right)$$

* **Why these weights?** Scent description ($0.45$) and main accords ($0.25$) drive $70\%$ of the similarity, while gender ($0.20$) and brand ($0.10$) act as soft structural preferences.
* **Why `faiss.IndexFlatIP`?** Because every vector is L2-normalized to unit length ($\|\mathbf{x}\|_2 = 1$), the inner product (dot product) computed by `IndexFlatIP` is mathematically identical to **cosine similarity**, enabling exact search across $70,000+$ perfumes in a few milliseconds.

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

W_DESC   = 0.45
W_ACCORD = 0.25
W_GENDER = 0.20
W_BRAND  = 0.10

final_embs = (
    W_DESC   * desc_embs +
    W_ACCORD * accord_embs +
    W_GENDER * gender_embs +
    W_BRAND  * brand_embs
)

final_embs = normalize(final_embs).astype(np.float32)

dim = final_embs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(final_embs)

# 3. Brand Lookup Table & Neutral Brand Fallback

* We build a lookup dictionary mapping each brand name to its normalized 384D brand embedding.
* Since `data.index` was reset to `0..N-1`, `data.index[data["Brand"] == b][0]` is guaranteed to be a valid positional index into `brand_embs`.
* If a user query does not mention any specific brand, we fall back to `mean_brand_emb` (the normalized centroid of all brand embeddings) so no particular brand is unfairly boosted.

In [ ]:
brands = sorted(data["Brand"].dropna().unique())
# Sort brands longest-first for regex matching so longer names match before shorter substrings
brands_by_length = sorted(brands, key=len, reverse=True)
brand_emb_dict = {}

for b in brands:
    idx = int(data.index[data["Brand"] == b][0])
    vec = brand_embs[idx]
    norm = np.linalg.norm(vec)
    brand_emb_dict[b] = (vec / norm) if norm > 0 else vec

mean_brand_emb = brand_embs.mean(axis=0)
mean_brand_emb = (mean_brand_emb / np.linalg.norm(mean_brand_emb)).astype(np.float32)

# 4. Query Parsing, Embedding & Recommendation Engine

### Key Improvements in Query Processing
1. **Word-Boundary Regex Matching (`\b`)**:
   * Previously, substring matching (`"men" in t` and `t.replace("men", "")`) matched `"men"` inside `"women"` and turned `"women"` into `"wo"`, and short brand tokens matched inside normal English words.
   * We now use `\b` word boundaries for both brand and gender detection/removal.
2. **Single SBERT Encoding per Query**:
   * The cleaned query text is encoded once with `model.encode([cleaned])` and shared between `desc_emb` and `accord_emb`, cutting query embedding time in half.
3. **Two-Stage Filtering & Ranking**:
   * **Stage 1 (Semantic Retrieval)**: Retrieve the `top_k` highest cosine-similarity candidates from FAISS and keep only those at or above the median similarity (`50th percentile`).
   * **Stage 2 (Quality Ranking)**: Sort the surviving semantically relevant candidates by their Bayesian `Weighted Rating` (`Score`).

In [ ]:
GENDER_CLEAN_PATTERN = re.compile(r"\b(unisex|women|woman|female|men|man|male)\b", flags=re.IGNORECASE)

def detect_brand_in_query(text, brands_list):
    """Detect if any known brand is mentioned as a whole word in the query text."""
    t = text.lower()
    for b in sorted(brands_list, key=len, reverse=True):
        if not b:
            continue
        if re.search(rf"\b{re.escape(b.lower())}\b", t):
            return b
    return None

def detect_gender_in_query(text):
    """Detect target gender from whole words in the query and return a 2D gender vector."""
    t = text.lower()
    if re.search(r"\bunisex\b", t):
        return UNISEX_VEC
    has_women = bool(re.search(r"\b(women|woman|female)\b", t))
    has_men = bool(re.search(r"\b(men|man|male)\b", t))
    if has_women and not has_men:
        return GENDER_VECTORS["women"]
    if has_men and not has_women:
        return GENDER_VECTORS["men"]
    return UNISEX_VEC

def clean_query_text(text, detected_brand=None):
    """Remove detected brand name and gender keywords using word boundaries so SBERT focuses on scent notes."""
    t = text.lower()
    if detected_brand:
        t = re.sub(rf"\b{re.escape(detected_brand.lower())}\b", " ", t)
    t = GENDER_CLEAN_PATTERN.sub(" ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t if t else text.strip()

def embed_query(text, model, brands, mean_brand_emb):
    # 1. Brand component
    brand = detect_brand_in_query(text, brands)
    brand_emb = brand_emb_dict[brand] if brand else mean_brand_emb

    # 2. Gender component
    gender_vec = detect_gender_in_query(text)
    gender_emb = expand_gender_vec(gender_vec)

    # 3. Clean text & encode once with SBERT for Description + Accords
    cleaned = clean_query_text(text, detected_brand=brand)
    text_emb = model.encode([cleaned])[0]
    text_emb = normalize(text_emb.reshape(1, -1))[0]

    desc_emb = text_emb
    accord_emb = text_emb
    brand_emb = normalize(brand_emb.reshape(1, -1))[0]
    gender_emb = normalize(gender_emb.reshape(1, -1))[0]

    # 4. Weighted Late Fusion
    q = (
        W_DESC   * desc_emb +
        W_ACCORD * accord_emb +
        W_GENDER * gender_emb +
        W_BRAND  * brand_emb
    )

    q = normalize(q.reshape(1, -1))[0]
    return q.astype(np.float32)

def recommend(query, model=model, brands=brands, mean_brand_emb=mean_brand_emb, faiss_index=index, data=data, top_k=10):
    q_emb = embed_query(query, model, brands, mean_brand_emb)

    D, I = faiss_index.search(q_emb.reshape(1, -1), top_k)

    results = []
    for idx, dist in zip(I[0], D[0]):
        idx = int(idx)
        row = data.iloc[idx]
        raw_row = raw_data.iloc[idx]
        results.append({
            "Name": row["Name"],
            "Brand": row["Brand"],
            "Gender": row["Gender"],
            "Main Accords": str(raw_row["Main Accords"]),
            "Description": str(raw_row["Description"]),
            "Score": row["Weighted Rating"],
            "Distance": float(dist),
            "URL": raw_row["url"],
        })

    results_df = pd.DataFrame(results)
    results_df["Main Accords"] = results_df["Main Accords"].apply(
        lambda x: x.replace("[]", "Not Specified")
    )
    results_df["Main Accords"] = results_df["Main Accords"].apply(
        lambda x: x.replace("]", "").replace("[", "").replace("'", "").replace(",", ", ")
    )

    # Keep top 50% by semantic similarity, then rank by Bayesian Weighted Rating
    bottom_limit = results_df["Distance"].quantile(0.5)
    results_df = results_df[results_df["Distance"] >= bottom_limit].sort_values("Score", ascending=False)
    return results_df, results

# 5. Test Query

In [ ]:
out, _ = recommend(
    "Top notes are Lime, Orange Blossom, Herbal Notes and Melon; middle notes are Basil and Clary Sage; base notes are Tangerine, Oakmoss and Musk"
)
out